In [8]:
import polars as pl
from zoneinfo import ZoneInfo
from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
load_dotenv()

ENERGY_PARQUET_PATH = 'data/processed/entsoe_energy.parquet'
WEATHER_PARQUET_PATH = 'data/processed/era5_weather.parquet'
HOLIDAY_PARQUET_PATH = 'data/processed/holidays.parquet'
TIMEZONE = ZoneInfo('Europe/Berlin')

# Statistical Methods

We will now start modelling the data. We will start with some autoregression and seasonal methods before moving on to more sophisticated ones.

We will fit three models each: 

 1. one for the period before the energy crisis,
 2. one for the period after,
 3. one for the entire dataset.

We will also split the data for each of the models into a test and training set.
The training set will 80 % of the data, the test will be 20 %.
For the third set we will do multiple train, test splits using an expanding window technique. 

In [26]:
lf_energy = pl.scan_parquet(ENERGY_PARQUET_PATH)

CRISIS_START = datetime(2021, 7, 1, tzinfo=TIMEZONE) # rouhgly
CRISIS_END = datetime(2023, 1, 1, tzinfo=TIMEZONE)

model1_data = lf_energy.filter(pl.col('time') <= CRISIS_START)
model2_data = lf_energy.filter(pl.col('time') >= CRISIS_END)
model3_data = [lf_energy.filter(pl.col('time') <= datetime(2020+i, 1, 1, tzinfo=TIMEZONE)) for i in range(6)]


def split_data(data: pl.LazyFrame, time_col = 'time', test_frac = 0.2) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    t_min, t_max = data.select(pl.col(time_col).min().alias('min'), pl.col(time_col).max().alias('max')).collect().row(0)
    span = t_max - t_min
    test_start = t_max - (span * test_frac)
    return (
        data.filter(pl.col(time_col) < test_start),
        data.filter((pl.col(time_col) >= test_start) & (pl.col(time_col) < t_max)),
    )

model1_split = split_data(model1_data)
model2_split = split_data(model2_data)
model3_split = list(map(split_data, model3_data))

## Autoregressive Methods
### ARIMA
### SARIMAX
### other simple ones
 - Markov regime switching model
 - self exciting threshold model
### AR on wether/mix and load, temperature
 - ensembles - multiple AR models trained on short and long periods [1]
 - LASSO/elastic net - alternative to ordinary least squares (OLS) / Residual Sum of Squares (RSS) also called LEAR [1]
 - stabilizing transformations: variance stabilizing instead of log (negative an 0 prices) [1]
## Timeseries Methods
## Metrics [1]
 - mean absolute error (MAE)
 - root mean square error (RMSE)
 - mean absolute percentage error (MAPE)
 - symmetric mean absolute percentage error (sMAPE)
 - relative mean absolute error (rMAE) against some naive approach - copying the price of the previous week at the same time.

## Sources
[1]	J. Lago, G. Marcjasz, B. De Schutter, und R. Weron, „Forecasting day-ahead electricity prices: A review of state-of-the-art algorithms, best practices and an open-access benchmark“, Applied Energy, Bd. 293, S. 116983, Juli 2021, doi: 10.1016/j.apenergy.2021.116983.
